In [0]:
from pyspark.sql import functions as F
# catalog = dbutils.widgets.get("catalog")
# schema_bronze = dbutils.widgets.get("bronze")
# catalog = "glpi"
# schema_bronze = "bronze"


In [0]:
RAW_PATH = "/Volumes/glpi/bronze/raw"

TABELAS = [
    "glpi_tickets",
    "glpi_users",
    "glpi_tickets_users",
    "glpi_tickets_tickets",
    "glpi_ticketsatisfactions",
]

for tabela in TABELAS:

    print(f"Lendo {tabela}...")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("quote", '"') # Configuração para a coluna content que possui html com ,
        .option("escape", '"') # parsing incorreto corrigido
        .option("multiLine", True)
        .csv(f"{RAW_PATH}/{tabela}.csv")
    )

    print(f"{df.count():,} registros")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"glpi.bronze.{tabela}")
    )
    
    print(f"{tabela} criada.\n")

display(spark.sql("""
SHOW TABLES IN glpi.bronze
"""))
